# WP4: Clinician Interviews and Prompt Library

This notebook documents the findings from two clinician interviews conducted as part of WP4,
and presents the resulting structured library of target PACS prompts.

## Interviews conducted
- **Interview 1**: Niklas Lackner, Medizinphysiker / Postdoc, UKER
- **Interview 2**: Nikola Milivojevic, Assistenzarzt, UKE

## Goals of WP4
1. Extract real-world query needs from clinicians and PACS users
2. Understand preferred wording styles and terminology
3. Understand ambiguity handling requirements
4. Turn these findings into a structured library of target PACS prompts

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Imports done.')

## 1. Key Findings from the Interviews

### Real-world query needs

Both interviewees identified a clear gap in current PACS systems: finding patients by
clinical findings rather than by administrative metadata (patient ID, date, accession number)
is currently not possible or extremely tedious.

Key use cases identified:
- **Simple finding search**: find all patients with a specific pathology (e.g. pleural effusion)
- **Combined finding search**: find patients with co-occurring findings (e.g. pleural effusion with peripheral contrast enhancement)
- **Research cohorts**: assemble cohorts of 20-30 patients for specific research questions
- **Longitudinal tracking**: track how a finding (e.g. pulmonary nodule) develops over time
- **Device/protocol filter**: find studies from a specific scanner or with specific acquisition parameters (Interview 1)

### Preferred wording styles

Both interviewees prefer **medical terminology** over everyday language, as it is more specific.
However, they acknowledge that non-medical users (e.g. student assistants) may use
everyday language, and the system should support both.

Key terminology findings:
- **Ground glass opacity** is preferred over GGO (abbreviation rarely used in practice)
- **Atelectasis** is preferred over collapse (collapse can also mean pneumothorax)
- **Fibrosis** and **scarring** are NOT synonymous and should be treated separately
- **Lymphadenopathy** vs. **enlarged lymph nodes**: not synonymous; lymphadenopathy implies multiple enlarged nodes
- **Emphysema** vs. **bulla**: not synonymous; emphysema consists of multiple bullae
- **Pleural effusion with contrast enhancement**: can indicate pleuritis, pleural empyema, or pleural carcinosis - these should NOT be treated as synonyms

### Ambiguity handling requirements

Both interviewees prefer **completeness over precision**: when a query is ambiguous,
the system should return all possible matches and let the user filter manually.
Missing relevant results is worse than returning too many.

Acceptable precision rates:
- Simple queries: 80-90% precision
- Specific/complex queries: ~50% precision (already a significant improvement over current systems)

## 2. Loading the Prompt Library

In [ ]:
with open('pacs_prompt_library.json', 'r', encoding='utf-8') as f:
    library = json.load(f)

prompts = library['prompts']
df = pd.DataFrame(prompts)

print(f'Prompt library loaded.')
print(f'  Total prompts:   {len(prompts)}')
print(f'  Categories:      {df["category"].nunique()}')
print(f'  From Interview 1: {sum(1 for p in prompts if "interview_1" in p["source"])}')
print(f'  From Interview 2: {sum(1 for p in prompts if "interview_2" in p["source"])}')
print(f'  From both:        {sum(1 for p in prompts if len(p["source"]) > 1)}')

## 3. Overview of the Prompt Library

In [ ]:
# print all prompts in a readable format
print('=== PACS PROMPT LIBRARY ===\n')

for category in library['metadata']['categories']:
    cat_prompts = [p for p in prompts if p['category'] == category]
    if not cat_prompts:
        continue

    print(f'\n--- {category.upper().replace("_", " ")} ({len(cat_prompts)} prompts) ---')

    for p in cat_prompts:
        source_str = ', '.join(p['source'])
        print(f"\n  [{p['id']}] Complexity: {p['complexity']} | Ambiguity: {p['ambiguity']} | Source: {source_str}")
        print(f"  DE: {p['de']}")
        print(f"  EN: {p['en']}")
        if p['notes']:
            print(f"  Notes: {p['notes']}")

## 4. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('WP4 - Prompt Library Overview', fontsize=13, fontweight='bold')

colors_cat  = plt.cm.tab10(np.linspace(0, 1, df['category'].nunique()))
colors_comp = ['#02C39A', '#065A82', '#E63946']
colors_amb  = ['#02C39A', '#F4A261', '#E63946']

# Plot 1: prompts per category
ax = axes[0]
cat_counts = df['category'].value_counts()
cat_labels = [c.replace('_', '\n') for c in cat_counts.index]
ax.barh(cat_labels, cat_counts.values, color=colors_cat, alpha=0.85)
ax.set_title('Prompts per Category', fontweight='bold')
ax.set_xlabel('Number of prompts')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Plot 2: complexity distribution
ax = axes[1]
comp_counts = df['complexity'].value_counts()
ax.pie(comp_counts.values, labels=comp_counts.index,
       autopct='%1.0f%%', colors=colors_comp, startangle=90)
ax.set_title('Complexity Distribution', fontweight='bold')

# Plot 3: ambiguity distribution
ax = axes[2]
amb_counts = df['ambiguity'].value_counts()
ax.pie(amb_counts.values, labels=amb_counts.index,
       autopct='%1.0f%%', colors=colors_amb, startangle=90)
ax.set_title('Ambiguity Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'prompt_library_overview.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR}/prompt_library_overview.png')

## 5. Metadata Requirements

Each prompt requires specific metadata fields from the CT-RATE dataset.
Here we analyse which fields are most commonly needed across all prompts.

In [ ]:
from collections import Counter

# count how often each metadata field is required
all_fields = [field for p in prompts for field in p['metadata_required']]
field_counts = Counter(all_fields)

print('=== METADATA REQUIREMENTS ===\n')
print(f'{"Field":40s} | {"Required by":10s} | {"% of prompts":12s}')
print('-' * 68)
for field, count in field_counts.most_common():
    pct = count / len(prompts) * 100
    print(f'{field:40s} | {count:10d} | {pct:10.1f}%')

# visualize
fig, ax = plt.subplots(figsize=(10, 5))
fields  = [f for f, c in field_counts.most_common()]
counts  = [c for f, c in field_counts.most_common()]
colors  = ['#065A82' if c == max(counts) else '#5B9BD5' for c in counts]

ax.barh(fields[::-1], counts[::-1], color=colors[::-1], alpha=0.85)
ax.set_title('Most required metadata fields across all prompts',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Number of prompts requiring this field')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'metadata_requirements.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIGURES_DIR}/metadata_requirements.png')

## 6. Terminology Mapping

Based on the interviews, we document the preferred terminology for each concept
and how it relates to our WP3 normalisation mappings.

In [ ]:
# terminology preferences from the interviews
# format: concept -> {preferred, synonyms, notes, wp3_canonical}
TERMINOLOGY = {
    'ground_glass_opacity': {
        'preferred_de':  'Milchglastrübung',
        'preferred_en':  'ground glass opacity',
        'avoid':         ['GGO'],
        'notes':         'GGO rarely used in practice (at least in Erlangen/Hamburg). '
                         'Milchglastrübung is a subtype of infiltrate.',
        'wp3_canonical': 'ground_glass_opacity'
    },
    'atelectasis': {
        'preferred_de':  'Atelektase',
        'preferred_en':  'atelectasis',
        'synonyms':      ['Kollaps / collapse (use with caution: can also mean pneumothorax)',
                          'Dystelektase (smaller atelectasis)',
                          'Minderbelüftung'],
        'avoid':         ['collapse (ambiguous)'],
        'notes':         'Collapse can also be interpreted as pneumothorax. '
                         'Atelectasis is preferred.',
        'wp3_canonical': 'atelectasis'
    },
    'fibrosis': {
        'preferred_de':  'Fibrose',
        'preferred_en':  'fibrosis',
        'not_synonymous': ['Vernarbung / scarring'],
        'notes':         'Fibrosis is used for underlying fibrotic lung disease (even if inactive). '
                         'Scarring is used for post-operative conditions. '
                         'These must NOT be treated as synonyms.',
        'wp3_canonical': 'fibrosis'
    },
    'lymphadenopathy': {
        'preferred_de':  'pathologisch vergrößerte Lymphknoten',
        'preferred_en':  'enlarged lymph nodes',
        'avoid':         ['LAP (unknown to both interviewees)'],
        'notes':         'Lymphadenopathy implies a higher number of enlarged nodes. '
                         'Not synonymous with a single enlarged lymph node.',
        'wp3_canonical': 'lymphadenopathy'
    },
    'emphysema': {
        'preferred_de':  'Emphysem',
        'preferred_en':  'emphysema',
        'not_synonymous': ['Bulla / bullae'],
        'notes':         'Emphysema consists of multiple bullae. '
                         'A single bulla is NOT emphysema.',
        'wp3_canonical': 'emphysema'
    },
    'pleural_effusion_with_enhancement': {
        'preferred_de':  'Pleuraerguss mit randständiger Kontrastmittelaufnahme',
        'preferred_en':  'pleural effusion with peripheral contrast enhancement',
        'not_synonymous': ['Pleuritis', 'Pleuraempyem', 'Pleurakarzinose'],
        'notes':         'These three diagnoses can present with the same CT finding '
                         'but are clinically distinct and must NOT be treated as synonyms.',
        'wp3_canonical': 'pleural_effusion'
    },
}

print('=== TERMINOLOGY PREFERENCES FROM INTERVIEWS ===\n')
for concept, info in TERMINOLOGY.items():
    print(f'CONCEPT: {concept}')
    print(f'  Preferred (DE): {info.get("preferred_de", "-")}')
    print(f'  Preferred (EN): {info.get("preferred_en", "-")}')
    if 'synonyms' in info:
        print(f'  Synonyms:       {", ".join(info["synonyms"])}')
    if 'not_synonymous' in info:
        print(f'  NOT synonymous: {", ".join(info["not_synonymous"])}')
    if 'avoid' in info:
        print(f'  Avoid:          {", ".join(info["avoid"])}')
    print(f'  Notes:          {info["notes"]}')
    print(f'  WP3 canonical:  {info["wp3_canonical"]}')
    print()

## 7. How WP4 Findings Support the LLM System

The interview findings directly inform several components of the LLM-based PACS query system:

**1. Query understanding**  
The prompt library provides validated examples of real-world queries that the LLM must understand.
These cover a range of complexity levels and ambiguity scenarios.

**2. Terminology normalisation (WP3 connection)**  
The terminology preferences identified in the interviews confirm and refine our WP3 mappings:
- Some mappings we defined are NOT valid (e.g. fibrosis ≠ scarring, emphysema ≠ bulla)
- Abbreviations like GGO and LAP should not be assumed as inputs
- The system must handle both German and English queries

**3. Ambiguity handling**  
Both interviewees prefer completeness over precision:
- Return all matches, let the user filter
- 50% precision is acceptable for complex queries
- 80-90% precision expected for simple queries

**4. Metadata requirements**  
The most critical metadata fields across all prompts are:
- `Findings_EN` (required by almost all prompts)
- `StudyDate` (for temporal filtering)
- `PatientAge` / `PatientSex` (for demographic filtering)
- Technical fields: `SliceThickness`, `ConvolutionKernel` (for Interview 1 use cases)

In [ ]:
print('=== WP4 SUMMARY ===')
print(f'\nInterviews conducted:     2')
print(f'Prompts in library:       {len(prompts)}')
print(f'Categories covered:       {df["category"].nunique()}')
print(f'\nComplexity breakdown:')
for level, count in df['complexity'].value_counts().items():
    print(f'  {level:10s}: {count}')
print(f'\nAmbiguity breakdown:')
for level, count in df['ambiguity'].value_counts().items():
    print(f'  {level:10s}: {count}')
print(f'\nKey terminology findings:  {len(TERMINOLOGY)}')
print(f'Unique metadata fields:    {len(field_counts)}')
print(f'\nWP4 complete!')